# Qwen3.5-4B — Random 200 / transcript only

Chọn ngẫu nhiên 200 mẫu từ toàn bộ train với seed=42. Dùng audio_text cho user hiện tại và history; không chạy ASR, không fine-tune.
Bật Internet và GPU T4 x2, rồi Run All. Một model FP16 chia layer trên hai GPU; không tạo hai bản model. Dataset riêng tư cymeu4l0t/zaloai-dataset được tìm trong Input hoặc nạp bằng KaggleHub nếu chưa gắn.
BATCH_SIZE=4: generate cùng lúc tối đa 4 transcript, left padding; mỗi mẫu có history và ToolEnv riêng. Mẫu đã dừng được bỏ khỏi các lượt sau; batch cuối có thể nhỏ hơn 4. VRAM batch 4 chưa được kiểm chứng trên GPU.
Prompt mở rộng quy tắc history/sửa lời, điều kiện/ngoại lệ, chọn đối tượng, ngày giờ, schema và missing-info; mô tả tool có Kind/Returns. Chưa đo chất lượng model với prompt mới.
Kết quả: score_100 (F1 gated by state hash), hash_accuracy, tool_call_f1, statuses và trace từng mẫu.
Scorer local theo công thức công bố; fuzzy tokenizer có thể khác BTC. Không áp latency gate.
Danh sách ID, seed và SHA256 dataset được lưu trong artifacts. Mẫu lỗi vẫn tính trong mẫu số 200.
Chưa chạy GPU trước khi lưu notebook. Giữ notebook/output riêng tư vì có dữ liệu cuộc thi.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys, zipfile
PROFILE = "random200"  # smoke: 8 mẫu; validation: 245 IDs cố định; public: toàn bộ public test
SAMPLE_SIZE = 200  # random200: lấy ngẫu nhiên từ toàn bộ train, không thay thế
SEED = 42
MODE = "oracle"    # train: oracle hoặc asr; public bắt buộc asr
DATA_ROOT = ""    # thư mục chứa train/ và public_test/; rỗng = tự tìm
DATA_ZIP = ""     # tùy chọn ZIP cuộc thi; rỗng = tự tìm nếu chưa có thư mục
ASR_MODEL_DIR = ""  # bắt buộc khi MODE=asr
MODEL = "Qwen/Qwen3.5-4B"  # hoặc đường dẫn snapshot đã gắn vào Kaggle
REVISION = "main"  # có thể đặt commit HF; SHA resolved được ghi vào config.json
MAX_INPUT = 8192
MAX_NEW_TOKENS = 768
MAX_STEPS = 8
NUM_GPUS = 2
BATCH_SIZE = 4     # tối đa 4 mẫu đang hoạt động trong mỗi lượt generate
RUN_NAME = "qwen35_random200_2gpu_batch4_policy_01"  # đổi tên khi chạy lại; không reset môi trường ngầm
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
WORK = Path("/kaggle/working/qwen35_baseline")
WORK.mkdir(exist_ok=True)


In [ ]:
PAYLOAD = {'agent.py': '"""Single-model LifeMate baseline. Standard library only; no hidden state input."""\nimport copy\nimport json\nimport re\nimport time\n\nSYSTEM = """You are LifeMate, a Vietnamese personal assistant.\n\nRequest and history:\n- Execute only the current user request. Historical actions are already applied\n  to the environment; never replay them just because they appear in history.\n- Resolve references such as "that one", "the other one", and "same as before"\n  from the conversation and actual tool results. Reuse established IDs and values\n  when still applicable; look up again only when relevant information is missing,\n  stale, or contradicted. An explicit new request to repeat an action is new work.\n- Apply the user\'s latest correction to the affected task; keep other requested\n  tasks. A withdrawn action must not execute. Never infer an unrequested action.\n\nConditions and selection:\n- Separate independent tasks from tasks that depend on earlier results. Obtain\n  evidence before evaluating a condition or carrying out a dependent action.\n- Honor negations, exceptions, limits, and fallback choices. If a condition is\n  false, skip only its dependent actions and continue other requested work.\n- For "all except...", inspect enough data to apply every filter and exclusion,\n  including relationships across objects or apps. Act on each matching object.\n- Match candidates using all supplied attributes and history. Do not pick the\n  first result arbitrarily. Resolve ambiguity with available evidence or lookup;\n  if it remains unresolved, ask for the required identifying parameter.\n- For "latest", "earliest", "cheapest", or similar comparisons, compare actual\n  fields, timestamps, and constraints; do not assume list order. Derive amounts,\n  addresses, or other values from observed data when the request refers to them.\n\nTime and context:\n- Use context.current_time and timezone, not the computer clock, for relative\n  dates and times. Use location and user_profile for references to home or work.\n- Compute offsets and intervals exactly, including crossing midnight. When moving\n  an event, preserve its duration unless the user changes the duration or end.\n- Respect schema formats: date YYYY-MM-DD, time HH:MM, date-time ISO-8601 with\n  the context timezone where applicable. Do not invent a required time from a\n  date alone. Use a recurrence parameter only if the tool supports it.\n\nTool contract:\n- Use only supplied tools, including unfamiliar ones, according to their\n  descriptions, Kind, Returns, and parameter schemas. read retrieves information;\n  write changes app data; meta reports missing information.\n- Include all required parameters. x-one-of requires at least one complete group\n  in addition to required; it does not require every alternative group. Prefer a\n  supported alternative already known, such as an explicit phone number over an\n  unnecessary contact lookup. x-ref identifies the source of an ID: use a supplied\n  or previously observed valid ID, or obtain it through the appropriate lookup.\n  Never invent IDs or tool results.\n- Follow types, enums, formats, bounds, and array item constraints. Numbers must\n  be numeric values without units or thousands separators; phone numbers remain\n  strings with leading zeroes. Omit unknown optional parameters rather than\n  inventing values. Preserve supplied content and meaning; x-match: fuzzy does\n  not permit adding facts or changing the requested message.\n- For updates, change only requested fields and fields needed to keep the\n  requested change consistent. Do not reset unrelated data or invent defaults.\n\nExecution and completion:\n- Choose exactly ONE native tool call per response, with no text after the call,\n  then wait for its actual result. Read the result before choosing the next step.\n- ok=true means the tool ran, not necessarily that the requested action succeeded.\n  Check done, returned objects, and messages. Empty results, done=false, or an\n  error are not success. Continue only with steps supported by observed evidence;\n  never repeat a failed write unchanged or fabricate a successful outcome.\n- Complete all independent supported tasks that can be resolved before asking\n  about a blocked task. First use context, history, and relevant lookup tools to\n  resolve missing values. Never ask for information already available there.\n- If a supported task still lacks a required value or has an ambiguous target,\n  call sys.ask_missing_info with the target tool_name and unresolved parameter\n  names exactly as in its schema, including the chosen x-one-of alternative.\n  Ask only for unresolved values. This meta call immediately ends the episode.\n- An unavailable capability is unsupported, not a missing parameter of a made-up\n  tool. Complete supported parts; do not call an unrelated tool as a substitute.\n- Do not repeat completed actions or make unnecessary lookup or verification\n  calls. Once no executable task remains, answer briefly without a tool call.\n  For a fully cancelled, unsupported, or no-action request, no tool call is needed.\n"""\n\ndef tools_for(item, catalog):\n    installed = {name for app in item["context"]["installed_apps"].values()\n                 for name in app["tools"]}\n    installed.add("sys.ask_missing_info")\n    tools = {}\n    for app in catalog["apps"]:\n        for tool in app["tools"]:\n            if tool["name"] in installed:\n                tools[tool["name"]] = tool\n    unknown = installed - tools.keys()\n    if unknown:\n        raise ValueError(f"Missing catalog tools: {sorted(unknown)}")\n    return tools\n\ndef native_tools(tools):\n    # Retain custom x-ref/x-one-of/x-match hints for the model and BTC validator.\n    functions = []\n    for tool in tools.values():\n        description = tool["description"]\n        for field, label in (("kind", "Kind"), ("returns", "Returns")):\n            if tool.get(field):\n                description += f"\\n{label}: {tool[field]}"\n        functions.append({"type": "function", "function": {\n            "name": tool["name"], "description": description,\n            "parameters": tool["parameters"]}})\n    return functions\n\ndef transcript(message, mode, cache):\n    if mode == "oracle":\n        return message["audio_text"]\n    text = cache[message["audio"]]\n    if not isinstance(text, str) or not text.strip():\n        raise ValueError(f"Empty/invalid transcript: {message[\'audio\']}")\n    return text\n\ndef call_message(call):\n    return {"role": "assistant", "content": "", "tool_calls": [\n        {"type": "function", "function": copy.deepcopy(call)}]}\n\ndef result_message(result):\n    return {"role": "tool", "content": json.dumps(result, ensure_ascii=False)}\n\ndef messages_for(item, mode="oracle", cache=None):\n    messages = [{"role": "system", "content": SYSTEM + "\\nContext:\\n" +\n                 json.dumps(item["context"], ensure_ascii=False)}]\n    for turn in item.get("history", []):\n        if turn["role"] == "user":\n            messages.append({"role": "user", "content": transcript(turn, mode, cache)})\n        else:\n            calls, results = turn.get("tool_calls", []), turn.get("tool_results", [])\n            if len(calls) != len(results):\n                raise ValueError("History tool calls/results are not aligned")\n            for call, result in zip(calls, results):\n                messages.extend([call_message(call), result_message(result)])\n            if turn.get("text"):\n                messages.append({"role": "assistant", "content": turn["text"]})\n    messages.append({"role": "user", "content": transcript(item, mode, cache)})\n    return messages\n\ndef parse_action(text, tools):\n    """Parse Qwen3.5 native XML. No repairs or guesses of malformed arguments."""\n    text = text.strip()\n    # Non-thinking prompting does not guarantee the model omits this wrapper.\n    # Discard only one complete leading block; never execute calls inside it.\n    if text.startswith("<think>"):\n        thinking = re.match(r"<think>(.*?)</think>\\s*", text, re.S)\n        if not thinking or "<think>" in thinking.group(1):\n            raise ValueError("Malformed or incomplete thinking block")\n        text = text[thinking.end():].strip()\n        if not text:\n            raise ValueError("Empty answer after thinking block")\n    if "<think>" in text or "</think>" in text:\n        raise ValueError("Unexpected thinking marker outside leading block")\n    if "<tool_call>" not in text:\n        if any(tag in text for tag in ("tool_call", "<function=", "<parameter=")):\n            raise ValueError("Malformed tool call")\n        return None\n    blocks = re.findall(r"<tool_call>(.*?)</tool_call>", text, re.S)\n    if len(blocks) != 1 or text.count("<tool_call>") != 1:\n        raise ValueError("Expected exactly one complete tool call")\n    prefix, suffix = text.split("<tool_call>", 1)\n    suffix = suffix.split("</tool_call>", 1)[1]\n    if suffix.strip():\n        raise ValueError("Unexpected text after tool call")\n    match = re.fullmatch(r"\\s*<function=([^>\\s]+)>(.*?)</function>\\s*", blocks[0], re.S)\n    if not match:\n        raise ValueError("Malformed function")\n    name, body = match.groups()\n    if name not in tools:\n        raise ValueError(f"Unavailable tool: {name}")\n    properties = tools[name]["parameters"].get("properties", {})\n    args = {}\n    pattern = r"<parameter=([^>\\s]+)>(.*?)</parameter>"\n    if re.sub(pattern, "", body, flags=re.S).strip():\n        raise ValueError("Malformed parameters")\n    for key, raw in re.findall(pattern, body, re.S):\n        if key in args or key not in properties:\n            raise ValueError(f"Duplicate/unknown parameter: {key}")\n        # Remove only the template\'s framing newlines for strings.\n        value = raw.removeprefix("\\n").removesuffix("\\n")\n        kind = properties[key].get("type")\n        args[key] = value if kind == "string" else json.loads(value)\n    return {"name": name, "arguments": args}\n\ndef _episode(item, catalog, env, mode, cache, max_steps):\n    started = time.monotonic()\n    tools = tools_for(item, catalog)\n    messages = messages_for(item, mode, cache)\n    trace, status = [], "step_limit"\n    error = None\n    for step in range(max_steps):\n        entry = {"step": step}\n        trace.append(entry)\n        try:\n            text = yield messages, native_tools(tools)\n            if isinstance(text, Exception):\n                raise text\n            entry["raw"] = text\n            call = parse_action(text, tools)\n            if call is None:\n                status = "stopped"\n                break\n            entry["action"] = call\n            check = env.validate(call["name"], call["arguments"])\n            if not check["ok"]:\n                raise ValueError(f"Schema rejected proposal: {check}")\n            result = env.call(call["name"], call["arguments"])\n            entry["observation"] = result\n            messages.extend([call_message(call), result_message(result)])\n            if call["name"] == "sys.ask_missing_info":\n                status = "missing_info"\n                break\n        except Exception as exc:\n            status, error = "error", f"{type(exc).__name__}: {exc}"\n            entry["error"] = error\n            break\n    return {"id": item["id"], "tool_calls": env.get_tool_calls(),\n            "state_hash": env.get_state_hash()}, {\n                "id": item["id"], "status": status, "error": error, "trace": trace,\n                "seconds": time.monotonic() - started}\n\ndef run_batch(items, catalog, envs, generate_batch, mode="oracle", cache=None, max_steps=8):\n    """Advance independent episodes together, removing finished rows each round."""\n    results = [None] * len(items)\n    active = {}\n    for i, (item, env) in enumerate(zip(items, envs, strict=True)):\n        episode = _episode(item, catalog, env, mode, cache, max_steps)\n        active[i] = (episode, next(episode))\n    while active:\n        try:\n            outputs = generate_batch([request for episode, request in active.values()])\n            if len(outputs) != len(active):\n                raise ValueError("Generator returned the wrong number of batch outputs")\n        except Exception as exc:\n            outputs = [exc] * len(active)\n        for i, output in zip(list(active), outputs):\n            episode, request = active[i]\n            try:\n                active[i] = (episode, episode.send(output))\n            except StopIteration as done:\n                results[i] = done.value\n                del active[i]\n    return results\n\ndef run_episode(item, catalog, env, generate, mode="oracle", cache=None, max_steps=8):\n    return run_batch([item], catalog, [env],\n                     lambda requests: [generate(*request) for request in requests],\n                     mode, cache, max_steps)[0]\n', 'run.py': '"""CLI: python qwen35_baseline/run.py --help."""\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport sys\nimport time\n\nfrom agent import messages_for, run_batch\nfrom scoring import score\n\nclass HFGenerator:\n    def __init__(self, model_id, revision, max_input, max_new_tokens, num_gpus=2):\n        import torch\n        from transformers import AutoModelForImageTextToText, AutoTokenizer\n        if not torch.cuda.is_available():\n            raise RuntimeError("CUDA GPU required; run the Kaggle notebook")\n        if torch.cuda.device_count() < num_gpus:\n            raise RuntimeError(f"Requested {num_gpus} GPUs; select GPU T4 x2 in Kaggle settings")\n        self.torch = torch\n        self.num_gpus = num_gpus\n        self.tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)\n        self.tokenizer.padding_side = "left"\n        if self.tokenizer.pad_token_id is None:\n            self.tokenizer.pad_token = self.tokenizer.eos_token\n        placement = {"device_map": {"": 0}} if num_gpus == 1 else {\n            "device_map": "balanced", "max_memory": {0: "10GiB", 1: "10GiB"}}\n        self.model = AutoModelForImageTextToText.from_pretrained(\n            model_id, revision=revision, dtype=torch.float16,\n            attn_implementation="sdpa", **placement).eval()\n        self.device_map = {key: str(value) for key, value in self.model.hf_device_map.items()}\n        used = {str(i) for i in range(num_gpus)}\n        actual = {value.removeprefix("cuda:") for value in self.device_map.values()}\n        if actual != used:\n            raise RuntimeError(f"Model must span GPUs {used}, without CPU/disk offload: {self.device_map}")\n        self.input_device = self.model.get_input_embeddings().weight.device\n        print(json.dumps({"stage": "model_loaded", "num_gpus": num_gpus,\n                          "device_map": self.device_map,\n                          "input_device": str(self.input_device)}), flush=True)\n        self.max_input, self.max_new_tokens = max_input, max_new_tokens\n        self.stats = []\n        self.resolved_revision = getattr(self.model.config, "_commit_hash", None)\n\n    def __call__(self, messages, tools):\n        result = self.generate_batch([(messages, tools)])[0]\n        if isinstance(result, Exception):\n            raise result\n        return result\n\n    def generate_batch(self, requests):\n        results = [None] * len(requests)\n        encoded, indices = [], []\n        for i, (messages, tools) in enumerate(requests):\n            try:\n                prompt = self.tokenizer.apply_chat_template(\n                    messages, tools=tools, tokenize=False,\n                    add_generation_prompt=True, enable_thinking=False)\n                tokens = self.tokenizer(prompt, add_special_tokens=False)\n                length = len(tokens["input_ids"])\n                if length > self.max_input:\n                    raise ValueError(f"Input {length} tokens exceeds {self.max_input}; no truncation")\n                encoded.append(tokens)\n                indices.append(i)\n            except Exception as exc:\n                results[i] = exc\n        if not encoded:\n            return results\n        try:\n            decoded = self._generate_encoded(encoded)\n            for i, result in zip(indices, decoded, strict=True):\n                results[i] = result\n        except Exception as exc:\n            for i in indices:\n                results[i] = exc.with_traceback(None)\n        return results\n\n    def _generate_encoded(self, encoded):\n        inputs = self.tokenizer.pad(encoded, padding=True, return_tensors="pt")\n        length = inputs["input_ids"].shape[-1]\n        inputs = {k: v.to(self.input_device) for k, v in inputs.items()}\n        started = time.monotonic()\n        with self.torch.inference_mode():\n            output = self.model.generate(\n                **inputs, do_sample=False, max_new_tokens=self.max_new_tokens,\n                pad_token_id=self.tokenizer.pad_token_id, logits_to_keep=1)\n        rows = output[:, length:].tolist()\n        eos = self.model.generation_config.eos_token_id\n        eos = [eos] if isinstance(eos, int) else (eos or [])\n        results, output_lengths = [], []\n        for tokens in rows:\n            stop = next((i + 1 for i, token in enumerate(tokens) if token in eos), None)\n            output_lengths.append(stop or len(tokens))\n            if stop is None:\n                results.append(ValueError("Generation did not reach EOS; no truncated action executed"))\n            else:\n                results.append(self.tokenizer.decode(tokens[:stop], skip_special_tokens=True))\n        self.stats.append({"batch_size": len(encoded),\n                           "input_tokens": [len(row["input_ids"]) for row in encoded],\n                           "padded_input_tokens": length, "output_tokens": output_lengths,\n                           "seconds": time.monotonic() - started})\n        return results\n\ndef read_rows(root, split, ids=None, limit=None):\n    rows = [json.loads(line) for line in (root/split/f"{split}.jsonl").read_text().splitlines()\n            if line.strip()]\n    if ids:\n        selected = set(Path(ids).read_text().split())\n        available = {r["id"] for r in rows}\n        if selected - available:\n            raise ValueError(f"Unknown selected IDs: {sorted(selected - available)}")\n        rows = [r for r in rows if r["id"] in selected]\n    return rows[:limit] if limit else rows\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument("--data-root", type=Path, required=True)\n    p.add_argument("--split", choices=["train", "public_test"], default="train")\n    p.add_argument("--ids", type=Path)\n    p.add_argument("--limit", type=int)\n    p.add_argument("--mode", choices=["oracle", "asr"], default="oracle")\n    p.add_argument("--transcripts", type=Path)\n    p.add_argument("--model", default="Qwen/Qwen3.5-4B")\n    p.add_argument("--revision", default="main")\n    p.add_argument("--max-input", type=int, default=16384)\n    p.add_argument("--max-new-tokens", type=int, default=768)\n    p.add_argument("--max-steps", type=int, default=8)\n    p.add_argument("--num-gpus", type=int, choices=[1, 2], default=2)\n    p.add_argument("--batch-size", type=int, default=4)\n    p.add_argument("--output", type=Path, required=True)\n    a = p.parse_args()\n    if a.split != "train" and a.mode == "oracle":\n        p.error("Oracle transcripts exist only in train; public_test requires --mode asr")\n    if a.mode == "asr" and not a.transcripts:\n        p.error("--mode asr requires --transcripts")\n    if min(a.max_input, a.max_new_tokens, a.max_steps, a.batch_size) < 1 or (a.limit is not None and a.limit < 1):\n        p.error("Token, step, batch and sample budgets must be positive")\n    root = a.data_root.resolve()\n    rows = read_rows(root, a.split, a.ids, a.limit)\n    if not rows:\n        p.error("No selected samples")\n    cache = json.loads(a.transcripts.read_text()) if a.transcripts else None\n    # Check every transcript before allocating the model or opening environments.\n    for row in rows:\n        messages_for(row, a.mode, cache)\n    catalog_file = root/a.split/f"{a.split}_tools_catalog.json"\n    catalog = json.loads(catalog_file.read_text())\n    scoring_catalog = {t["name"]: t for app in catalog["apps"] for t in app["tools"]}\n    # Import exactly the BTC package for this split in this fresh CLI process.\n    sys.path.insert(0, str(root/a.split))\n    from zaic_tools import ToolEnv, state_hash\n    if a.output.exists():\n        p.error("Output already exists; choose a fresh directory (no implicit reset/resume)")\n    a.output.mkdir(parents=True)\n    generator = HFGenerator(a.model, a.revision, a.max_input, a.max_new_tokens, a.num_gpus)\n    config = {k: str(v) if isinstance(v, Path) else v for k, v in vars(a).items()}\n    config.update(resolved_revision=generator.resolved_revision,\n                  catalog_sha256=hashlib.sha256(catalog_file.read_bytes()).hexdigest(),\n                  selected_ids=[r["id"] for r in rows],\n                  packages={n: importlib.metadata.version(n) for n in\n                            ("torch", "transformers", "accelerate")},\n                  gpus=[generator.torch.cuda.get_device_name(i) for i in range(a.num_gpus)],\n                  device_map=generator.device_map,\n                  input_device=str(generator.input_device))\n    (a.output/"config.json").write_text(json.dumps(config, indent=2))\n    summaries = []\n    started = time.monotonic()\n    with (a.output/"submission.jsonl").open("w") as submissions, \\\n         (a.output/"traces.jsonl").open("w") as traces:\n        for offset in range(0, len(rows), a.batch_size):\n            batch = rows[offset:offset + a.batch_size]\n            envs = [ToolEnv(a.split, row["id"], root=root) for row in batch]\n            results = run_batch(batch, catalog, envs, generator.generate_batch,\n                                a.mode, cache, a.max_steps)\n            for i, (row, (prediction, detail)) in enumerate(zip(batch, results), offset + 1):\n                # Labels are inspected only AFTER inference. Never supplied to the agent.\n                if a.split == "train":\n                    initial = json.loads((root/a.split/f"{a.split}_environment"/row["id"]/"state.json").read_text())\n                    initial.update(row["label"]["final_state"])\n                    reference = {"id": row["id"], "tool_calls": row["label"]["tool_calls"],\n                                 "state_hash": state_hash(initial)}\n                    metrics = score(prediction, reference, scoring_catalog)\n                    detail.update({key: metrics[key] for key in\n                                   ("score", "hash_match", "tool_f1", "matched", "pred_calls", "gold_calls")})\n                    detail["canonical_state_match"] = metrics["hash_match"]\n                    detail["exact_call_sequence_match"] = prediction["tool_calls"] == row["label"]["tool_calls"]\n                submissions.write(json.dumps(prediction, ensure_ascii=False) + "\\n")\n                traces.write(json.dumps(detail, ensure_ascii=False) + "\\n")\n                submissions.flush()\n                traces.flush()\n                summaries.append({k: v for k, v in detail.items() if k != "trace"})\n                print(json.dumps({"completed": i, "total": len(rows), **summaries[-1]}), flush=True)\n    from collections import Counter\n    report = {"samples": len(rows), "mode": a.mode, "batch_size": a.batch_size,\n              "seconds": time.monotonic()-started,\n              "statuses": dict(Counter(r["status"] for r in summaries)),\n              "note": "Local approximation of published hash-gated F1; fuzzy tokenization may differ from BTC. No latency gate; model load and ASR excluded.",\n              "generation": generator.stats,\n              "peak_cuda_gib_per_gpu": {\n                  str(i): generator.torch.cuda.max_memory_allocated(i)/2**30\n                  for i in range(a.num_gpus)}}\n    if a.split == "train":\n        for metric in ("canonical_state_match", "exact_call_sequence_match"):\n            report[metric] = sum(r[metric] for r in summaries)/len(rows)\n        report["score_100"] = 100 * sum(r["score"] for r in summaries)/len(rows)\n        report["hash_accuracy"] = sum(r["hash_match"] for r in summaries)/len(rows)\n        report["hash_accuracy_percent"] = 100 * report["hash_accuracy"]\n        report["tool_call_f1"] = sum(r["tool_f1"] for r in summaries)/len(rows)\n    (a.output/"report.json").write_text(json.dumps(report, indent=2))\n\nif __name__ == "__main__":\n    main()\n', 'scoring.py': '"""Local approximation of published hash-gated tool-call F1 (not BTC scorer)."""\nfrom collections import Counter\nimport re\nimport unicodedata\n\ndef token_f1(a,b):\n    a=Counter(re.findall(r"\\w+",unicodedata.normalize("NFC",str(a)).lower()))\n    b=Counter(re.findall(r"\\w+",unicodedata.normalize("NFC",str(b)).lower()))\n    if not a and not b:return 1.\n    overlap=sum((a&b).values())\n    return 2*overlap/(sum(a.values())+sum(b.values())) if overlap else 0.\n\ndef matches(a,b,catalog):\n    if a.get("name")!=b.get("name"):return False\n    x,y=a.get("arguments",{}),b.get("arguments",{})\n    if x.keys()!=y.keys():return False\n    props=catalog.get(a["name"],{}).get("parameters",{}).get("properties",{})\n    for key,value in x.items():\n        if props.get(key,{}).get("x-match")=="fuzzy":\n            if token_f1(value,y[key])<=.6:return False\n        elif value!=y[key] or isinstance(value,bool)!=isinstance(y[key],bool):return False\n    return True\n\ndef maximum_matches(pred,gold,catalog):\n    edges=[[j for j,g in enumerate(gold) if matches(p,g,catalog)] for p in pred]\n    assigned={}\n    def augment(i,seen):\n        for j in edges[i]:\n            if j in seen:continue\n            seen.add(j)\n            if j not in assigned or augment(assigned[j],seen):\n                assigned[j]=i;return True\n        return False\n    return sum(augment(i,set()) for i in range(len(pred)))\n\ndef score(pred,gold,catalog,timed_out=False):\n    calls,ref=pred["tool_calls"],gold["tool_calls"]\n    matched=maximum_matches(calls,ref,catalog)\n    hash_ok=pred["state_hash"]==gold["state_hash"]\n    f1=2*matched/(len(calls)+len(ref)) if calls or ref else 1.\n    value=0. if timed_out or not hash_ok else f1\n    return {"id":gold["id"],"score":value,"hash_match":hash_ok,"tool_f1":f1,\n            "matched":matched,"pred_calls":len(calls),"gold_calls":len(ref),"timed_out":timed_out}\n', 'transcribe.py': '"""Transcribe selected current AND history user audio with supplied Zipformer."""\nimport argparse\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nfrom run import read_rows\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument("--data-root", type=Path, required=True)\n    p.add_argument("--split", default="train", choices=["train", "public_test"])\n    p.add_argument("--ids", type=Path)\n    p.add_argument("--limit", type=int)\n    p.add_argument("--model-dir", type=Path, required=True)\n    p.add_argument("--output", type=Path, required=True)\n    a = p.parse_args()\n    import numpy as np\n    import soundfile as sf\n    import sherpa_onnx\n    from scipy.signal import resample_poly\n    rows = read_rows(a.data_root, a.split, a.ids, a.limit)\n    names = list(dict.fromkeys(\n        name for row in rows for name in\n        [row["audio"]] + [m["audio"] for m in row["history"] if m["role"] == "user"]))\n    if not names:\n        raise ValueError("No audio selected")\n    paths = {key: a.model_dir/f"{key}.onnx" for key in ("encoder", "decoder", "joiner")}\n    paths["tokens"] = a.model_dir/"tokens.txt"\n    hashes = {key: hashlib.sha256(path.read_bytes()).hexdigest() for key, path in paths.items()}\n    recognizer = sherpa_onnx.OfflineRecognizer.from_transducer(\n        **{key: str(path) for key, path in paths.items()}, num_threads=4,\n        sample_rate=16000, feature_dim=80, decoding_method="greedy_search", provider="cpu")\n    if a.output.exists():\n        raise ValueError("Transcript output already exists; choose a fresh path")\n    a.output.parent.mkdir(parents=True, exist_ok=True)\n    texts = {}\n    for index, name in enumerate(names, 1):\n        base = (a.data_root/a.split).resolve()\n        path = (base/name).resolve()\n        if not path.is_relative_to(base):\n            raise ValueError("Audio path escapes split root")\n        samples, rate = sf.read(path, dtype="float32", always_2d=True)\n        samples = samples.mean(axis=1)\n        if not len(samples) or not np.isfinite(samples).all():\n            raise ValueError(f"Invalid audio: {name}")\n        if rate != 16000:\n            factor = math.gcd(rate, 16000)\n            samples = resample_poly(samples, 16000//factor, rate//factor)\n        stream = recognizer.create_stream()\n        stream.accept_waveform(16000, np.ascontiguousarray(samples, dtype=np.float32))\n        recognizer.decode_stream(stream)\n        texts[name] = stream.result.text.strip()\n        if not texts[name]:\n            raise ValueError(f"Empty ASR result: {name}")\n        print(f"ASR {index}/{len(names)} {name}", flush=True)\n    a.output.write_text(json.dumps(texts, ensure_ascii=False, indent=2))\n    a.output.with_suffix(".meta.json").write_text(json.dumps(\n        {"split": a.split, "selected_ids": [r["id"] for r in rows],\n         "model_sha256": hashes, "sherpa_onnx": sherpa_onnx.__version__,\n         "provider": "cpu", "current_and_history": True}, indent=2))\n\nif __name__ == "__main__":\n    main()\n', 'test_agent.py': 'import copy\nimport json\nimport unittest\nfrom agent import messages_for, native_tools, parse_action, run_batch, run_episode, tools_for\n\nCATALOG = {"apps": [{"tools": [\n    {"name": "contacts.search", "description": "Find", "parameters": {\n        "type": "object", "properties": {"query": {"type": "string"}}}},\n    {"name": "sms.send", "description": "Send", "parameters": {\n        "type": "object", "properties": {"contact_id": {"type": "string"},\n                                        "content": {"type": "string"}}}},\n    {"name": "sys.ask_missing_info", "description": "Ask", "parameters": {\n        "type": "object", "properties": {"tool_name": {"type": "string"},\n                                        "params": {"type": "array"}}}}\n]}]}\nITEM = {"id": "synthetic", "audio": "now.wav", "audio_text": "Nhắn Nam xin chào",\n        "history": [], "context": {"installed_apps": {\n            "contacts": {"tools": ["contacts.search"]},\n            "sms": {"tools": ["sms.send"]}}}}\nTOOLS = {t["name"]: t for t in CATALOG["apps"][0]["tools"]}\n\ndef xml(name, args):\n    return "<tool_call>\\n<function=" + name + ">\\n" + "".join(\n        "<parameter=" + k + ">\\n" +\n        (v if isinstance(v, str) else json.dumps(v)) + "\\n</parameter>\\n"\n        for k, v in args.items()) + "</function>\\n</tool_call>"\n\nclass Env:\n    def __init__(self):\n        self.calls = []\n    def validate(self, name, args):\n        return {"ok": not (name == "sms.send" and "contact_id" not in args)}\n    def call(self, name, args):\n        self.calls.append({"name": name, "arguments": args})\n        return {"ok": True, "contacts": [{"id": "observed_42"}]}\n    def get_tool_calls(self):\n        return copy.deepcopy(self.calls)\n    def get_state_hash(self):\n        return "hash"\n\nclass AgentTests(unittest.TestCase):\n    def test_tool_metadata_and_complete_schema_survive_native_conversion(self):\n        catalog = copy.deepcopy(CATALOG)\n        tool = {"name": "new_app.dispatch", "kind": "write", "returns": "receipt_id",\n                "description": "Dispatch to an observed target.", "parameters": {\n                    "type": "object", "required": ["amount"],\n                    "x-one-of": [["target_id"], ["phone"]], "properties": {\n                        "target_id": {"type": "string", "x-ref": "new_app.targets"},\n                        "phone": {"type": "string"},\n                        "amount": {"type": "integer", "exclusiveMinimum": 0},\n                        "note": {"type": "string", "x-match": "fuzzy"},\n                        "date": {"type": "string", "format": "date"},\n                        "tags": {"type": "array", "items": {"type": "string", "enum": ["a", "b"]}}}}}\n        catalog["apps"].append({"tools": [tool]})\n        item = copy.deepcopy(ITEM)\n        item["context"]["installed_apps"] = {"new_app": {"tools": [tool["name"]]}}\n        before = copy.deepcopy(catalog)\n        functions = {t["function"]["name"]: t["function"]\n                     for t in native_tools(tools_for(item, catalog))}\n        self.assertEqual(set(functions), {tool["name"], "sys.ask_missing_info"})\n        actual = functions[tool["name"]]\n        self.assertEqual(actual["parameters"], tool["parameters"])\n        self.assertEqual(actual["description"],\n                         tool["description"] + "\\nKind: write\\nReturns: receipt_id")\n        self.assertEqual(set(actual), {"name", "description", "parameters"})\n        self.assertEqual(functions["sys.ask_missing_info"]["description"], "Ask")\n        self.assertEqual(catalog, before)\n\n    def test_independent_action_then_empty_lookup_then_terminal_missing_info(self):\n        item = copy.deepcopy(ITEM)\n        item["audio_text"] = "Nhắn Nam xin chào, rồi gọi Minh."\n        item["history"] = [{"role": "assistant", "text": "Nam có ID known_nam."}]\n        empty = {"ok": True, "done": False, "contacts": [], "message": "No match"}\n        class EmptyLookupEnv(Env):\n            def call(self, name, args):\n                super().call(name, args)\n                return copy.deepcopy(empty) if name == "contacts.search" else {"ok": True}\n        env = EmptyLookupEnv()\n        def generate(messages, tools):\n            if not env.calls:\n                self.assertIn("known_nam", str(messages))\n                return xml("sms.send", {"contact_id": "known_nam", "content": "xin chào"})\n            if len(env.calls) == 1:\n                return xml("contacts.search", {"query": "Minh"})\n            self.assertEqual(json.loads(messages[-1]["content"]), empty)\n            return xml("sys.ask_missing_info", {"tool_name": "phone.call", "params": ["contact_id"]})\n        # Supply the future target tool without requiring any hardcoded app list.\n        catalog = copy.deepcopy(CATALOG)\n        catalog["apps"].append({"tools": [{"name": "phone.call", "description": "Call",\n            "parameters": {"type": "object", "properties": {"contact_id": {"type": "string"}}}}]})\n        item["context"]["installed_apps"]["phone"] = {"tools": ["phone.call"]}\n        prediction, detail = run_episode(item, catalog, env, generate)\n        self.assertEqual([call["name"] for call in prediction["tool_calls"]],\n                         ["sms.send", "contacts.search", "sys.ask_missing_info"])\n        self.assertEqual(detail["status"], "missing_info")\n        self.assertEqual(detail["trace"][1]["observation"], empty)\n\n    def test_observation_drives_next_action(self):\n        env = Env()\n        def generate(messages, tools):\n            self.assertNotIn("label", str(messages))\n            if not env.calls:\n                return xml("contacts.search", {"query": "Nam"})\n            if len(env.calls) == 1:\n                result = json.loads(messages[-1]["content"])\n                return xml("sms.send", {"contact_id": result["contacts"][0]["id"],\n                                        "content": "xin chào"})\n            return "Đã gửi."\n        pred, detail = run_episode(ITEM, CATALOG, env, generate)\n        self.assertEqual(pred["tool_calls"][1]["arguments"]["contact_id"], "observed_42")\n        self.assertEqual(detail["status"], "stopped")\n\n    def test_history_is_evidence_not_replayed(self):\n        item = copy.deepcopy(ITEM)\n        item["history"] = [{"role": "user", "audio": "old.wav", "audio_text": "Tìm Nam"},\n            {"role": "assistant", "text": "Đã tìm.", "tool_calls": [\n                {"name": "contacts.search", "arguments": {"query": "Nam"}}],\n             "tool_results": [{"ok": True, "contacts": [{"id": "old_id"}]}]}]\n        env = Env()\n        pred, _ = run_episode(item, CATALOG, env, lambda m, t: "Không cần thao tác.")\n        self.assertEqual(pred["tool_calls"], [])\n        self.assertIn("old_id", str(messages_for(item)))\n\n    def test_asr_uses_history_and_current_without_oracle_fallback(self):\n        item = copy.deepcopy(ITEM)\n        item["history"] = [{"role": "user", "audio": "old.wav", "audio_text": "ORACLE"}]\n        messages = messages_for(item, "asr", {"old.wav": "ASR old", "now.wav": "ASR now"})\n        self.assertEqual([m["content"] for m in messages[1:]], ["ASR old", "ASR now"])\n        with self.assertRaises(KeyError):\n            messages_for(item, "asr", {"now.wav": "now"})\n\n    def test_schema_rejection_does_not_execute(self):\n        env = Env()\n        pred, detail = run_episode(ITEM, CATALOG, env, lambda m, t: xml(\n            "sms.send", {"content": "hi"}))\n        self.assertEqual(pred["tool_calls"], [])\n        self.assertEqual(detail["status"], "error")\n\n    def test_missing_info_is_executed_and_terminal(self):\n        env = Env()\n        pred, detail = run_episode(ITEM, CATALOG, env, lambda m, t: xml(\n            "sys.ask_missing_info", {"tool_name": "sms.send", "params": ["contact_id"]}))\n        self.assertEqual(len(pred["tool_calls"]), 1)\n        self.assertEqual(detail["status"], "missing_info")\n\n    def test_reject_malformed_multiple_unknown_calls(self):\n        call = xml("contacts.search", {"query": "Nam"})\n        for raw in [call + call, call[:-4], call + "suffix",\n                    xml("unknown.tool", {}), "<tool_call>broken</tool_call>"]:\n            with self.subTest(raw=raw), self.assertRaises(ValueError):\n                parse_action(raw, TOOLS)\n\n    def test_complete_leading_thinking_before_call_or_answer(self):\n        call = xml("contacts.search", {"query": "Nam"})\n        expected = {"name": "contacts.search", "arguments": {"query": "Nam"}}\n        for block in ("<think>\\n\\n</think>", "<think>Chọn công cụ.\\n</think>"):\n            with self.subTest(block=block):\n                self.assertEqual(parse_action(block + "\\n" + call, TOOLS), expected)\n                self.assertIsNone(parse_action(block + "\\nĐã xong.", TOOLS))\n\n    def test_calls_inside_thinking_are_never_executed(self):\n        hidden = xml("sms.send", {"contact_id": "invented", "content": "hi"})\n        visible = xml("contacts.search", {"query": "Nam"})\n        env = Env()\n        outputs = iter(["<think>" + hidden + "</think>\\n" + visible, "Đã xong."])\n        pred, detail = run_episode(ITEM, CATALOG, env, lambda m, t: next(outputs))\n        self.assertEqual(pred["tool_calls"], [\n            {"name": "contacts.search", "arguments": {"query": "Nam"}}])\n        self.assertEqual(detail["status"], "stopped")\n        self.assertIn(hidden, detail["trace"][0]["raw"])\n\n    def test_reject_incomplete_nested_or_misplaced_thinking(self):\n        call = xml("contacts.search", {"query": "Nam"})\n        for raw in ("<think>" + call, "</think>" + call,\n                    "<think>outer<think>inner</think></think>" + call,\n                    "<think></think><think></think>" + call,\n                    "<think></think>", "prefix<think></think>" + call,\n                    call + "<think></think>"):\n            with self.subTest(raw=raw), self.assertRaises(ValueError):\n                parse_action(raw, TOOLS)\n\n    def test_thinking_does_not_relax_tool_validation(self):\n        call = xml("contacts.search", {"query": "Nam"})\n        for raw in (call + call, call[:-4], call + "suffix", xml("unknown.tool", {})):\n            with self.subTest(raw=raw), self.assertRaises(ValueError):\n                parse_action("<think>Chọn công cụ.</think>\\n" + raw, TOOLS)\n\n    def test_step_budget_preserves_executed_calls(self):\n        env = Env()\n        pred, detail = run_episode(ITEM, CATALOG, env, lambda m, t: xml(\n            "contacts.search", {"query": "Nam"}), max_steps=2)\n        self.assertEqual(len(pred["tool_calls"]), 2)\n        self.assertEqual(detail["status"], "step_limit")\n\n    def test_batch_shrinks_and_keeps_histories_tools_and_environments_separate(self):\n        items = [copy.deepcopy(ITEM) for _ in range(4)]\n        envs = [Env() for _ in items]\n        for i, item in enumerate(items):\n            item.update(id=f"row_{i}", audio_text=f"request_{i}")\n        del items[3]["context"]["installed_apps"]["sms"]\n        batch_sizes = []\n        def generate(requests):\n            batch_sizes.append(len(requests))\n            outputs = []\n            for messages, tools in requests:\n                user = next(m["content"] for m in messages if m["role"] == "user")\n                i = int(user.removeprefix("request_"))\n                self.assertNotIn("label", str(messages))\n                names = {t["function"]["name"] for t in tools}\n                self.assertEqual("sms.send" in names, i != 3)\n                if i == 0:\n                    output = "Đã xong."\n                elif i == 2:\n                    output = xml("sys.ask_missing_info", {\n                        "tool_name": "sms.send", "params": ["contact_id"]})\n                elif not envs[i].calls:\n                    output = xml("contacts.search", {"query": user})\n                elif i == 3:\n                    output = ValueError("row-specific generation failure")\n                elif len(envs[i].calls) == 1:\n                    self.assertEqual(json.loads(messages[-1]["content"])["contacts"][0]["id"],\n                                     "observed_42")\n                    self.assertEqual(envs[i].calls[0]["arguments"]["query"], user)\n                    self.assertNotIn("request_3", str(messages))\n                    output = xml("sms.send", {"contact_id": "observed_42", "content": "hi"})\n                else:\n                    output = "Đã gửi."\n                outputs.append(output)\n            return outputs\n        results = run_batch(items, CATALOG, envs, generate)\n        self.assertEqual(batch_sizes, [4, 2, 1])\n        self.assertEqual([p["id"] for p, d in results], [item["id"] for item in items])\n        self.assertEqual([d["status"] for p, d in results],\n                         ["stopped", "stopped", "missing_info", "error"])\n        self.assertEqual([len(p["tool_calls"]) for p, d in results], [0, 2, 1, 1])\n        self.assertIn("row-specific", results[3][1]["error"])\n\n    def test_batch_generation_failure_preserves_previously_executed_calls(self):\n        envs = [Env(), Env()]\n        def generate(requests):\n            if not envs[0].calls:\n                return [xml("contacts.search", {"query": "Nam"})] * len(requests)\n            raise RuntimeError("batch failure")\n        results = run_batch([ITEM, ITEM], CATALOG, envs, generate)\n        for prediction, detail in results:\n            self.assertEqual(len(prediction["tool_calls"]), 1)\n            self.assertEqual(detail["status"], "error")\n            self.assertIn("batch failure", detail["error"])\n\n    def test_partial_batch_and_per_episode_step_budget(self):\n        env = Env()\n        sizes = []\n        def generate(requests):\n            sizes.append(len(requests))\n            return [xml("contacts.search", {"query": "Nam"})] * len(requests)\n        results = run_batch([ITEM], CATALOG, [env], generate, max_steps=2)\n        self.assertEqual(sizes, [1, 1])\n        self.assertEqual(len(results[0][0]["tool_calls"]), 2)\n        self.assertEqual(results[0][1]["status"], "step_limit")\n\nif __name__ == "__main__":\n    unittest.main()\n', 'test_scoring.py': 'import unittest\nfrom scoring import matches, maximum_matches, score\n\nCATALOG = {"sms.send": {"parameters": {"properties": {\n    "contact_id": {"type": "string"},\n    "content": {"type": "string", "x-match": "fuzzy"}}}}}\n\ndef call(content="xin chào", contact="c1"):\n    return {"name": "sms.send", "arguments": {\n        "contact_id": contact, "content": content}}\n\ndef prediction(calls, hash_value="correct"):\n    return {"id": "synthetic", "tool_calls": calls, "state_hash": hash_value}\n\nclass ScoreTests(unittest.TestCase):\n    def test_correct_calls_with_wrong_hash_score_zero(self):\n        result = score(prediction([call()], "wrong"), prediction([call()]), CATALOG)\n        self.assertEqual(result["tool_f1"], 1)\n        self.assertEqual(result["score"], 0)\n        self.assertFalse(result["hash_match"])\n\n    def test_duplicate_predictions_cannot_reuse_gold(self):\n        result = score(prediction([call(), call()]), prediction([call()]), CATALOG)\n        self.assertEqual(result["matched"], 1)\n        self.assertAlmostEqual(result["score"], 2/3)\n\n    def test_fuzzy_threshold_is_strict_and_ids_are_exact(self):\n        self.assertFalse(matches(call("a b c d e"), call("a b c x y"), CATALOG))\n        self.assertTrue(matches(call("Xin CHÀO"), call("xin chào"), CATALOG))\n        self.assertFalse(matches(call(contact="c1"), call(contact="c2"), CATALOG))\n\n    def test_order_independent_maximum_matching(self):\n        pred = [call("a b c d e"), call("a b c d x")]\n        gold = [call("a b c d x"), call("a b c e y")]\n        self.assertEqual(maximum_matches(pred, gold, CATALOG), 2)\n\n    def test_empty_calls_and_no_matches(self):\n        self.assertEqual(score(prediction([]), prediction([]), CATALOG)["score"], 1)\n        self.assertEqual(score(prediction([call()]), prediction([]), CATALOG)["score"], 0)\n        self.assertEqual(score(prediction([]), prediction([call()]), CATALOG)["score"], 0)\n\nif __name__ == "__main__":\n    unittest.main()\n', 'test_placement.py': '"""Test dispatch and input routing without allocating a real model or GPU."""\nimport contextlib\nimport io\nimport sys\nfrom types import SimpleNamespace\nimport unittest\nfrom unittest.mock import MagicMock, patch\n\nfrom run import HFGenerator\n\n\nclass PlacementTests(unittest.TestCase):\n    def setUp(self):\n        self.model = MagicMock()\n        self.model.eval.return_value = self.model\n        self.model.hf_device_map = {"layers.0": 0, "layers.16": 1}\n        self.model.get_input_embeddings.return_value.weight.device = "cuda:1"\n        self.model.generation_config.eos_token_id = 99\n        self.model.generate.return_value.__getitem__.return_value.tolist.return_value = [[99]]\n        self.tokens = MagicMock()\n        self.tokens.shape = (1, 2)\n        self.tokenizer = MagicMock()\n        self.mask = MagicMock()\n        self.tokenizer.return_value = {"input_ids": [7, 8], "attention_mask": [1, 1]}\n        self.tokenizer.pad.return_value = {"input_ids": self.tokens, "attention_mask": self.mask}\n        self.tokenizer.pad_token_id = 0\n        self.tokenizer.decode.return_value = "Done"\n        self.factory = MagicMock()\n        self.factory.from_pretrained.return_value = self.model\n        tokenizer_factory = MagicMock()\n        tokenizer_factory.from_pretrained.return_value = self.tokenizer\n        self.cuda = SimpleNamespace(is_available=lambda: True, device_count=lambda: 2)\n        self.torch = SimpleNamespace(cuda=self.cuda, float16="float16",\n                                     inference_mode=contextlib.nullcontext)\n        self.modules = {"torch": self.torch, "transformers": SimpleNamespace(\n            AutoModelForImageTextToText=self.factory, AutoTokenizer=tokenizer_factory)}\n\n    def load(self, count=2):\n        with patch.dict(sys.modules, self.modules), contextlib.redirect_stdout(io.StringIO()):\n            return HFGenerator("model", "revision", 8192, 768, count)\n\n    def test_two_gpu_dispatch_and_embedding_input_device(self):\n        generator = self.load()\n        kwargs = self.factory.from_pretrained.call_args.kwargs\n        self.assertEqual(kwargs["device_map"], "balanced")\n        self.assertEqual(kwargs["max_memory"], {0: "10GiB", 1: "10GiB"})\n        self.assertEqual(kwargs["dtype"], "float16")\n        self.assertEqual(generator([], []), "Done")\n        self.tokens.to.assert_called_once_with("cuda:1")\n        self.mask.to.assert_called_once_with("cuda:1")\n        self.assertEqual(self.tokenizer.padding_side, "left")\n        generator.tokenizer.apply_chat_template.assert_called_once_with(\n            [], tools=[], tokenize=False, add_generation_prompt=True, enable_thinking=False)\n\n    def test_batch_generates_once_and_trims_each_row_at_first_eos(self):\n        generator = self.load()\n        rows = [[10, 99, 0], [11, 12, 99], [13, 14, 15], [99, 0, 0]]\n        self.model.generate.return_value.__getitem__.return_value.tolist.return_value = rows\n        self.tokenizer.decode.side_effect = lambda tokens, **kwargs: str(tokens)\n        results = generator.generate_batch([([], [])] * 4)\n        self.model.generate.assert_called_once()\n        self.assertEqual(results[:2], ["[10, 99]", "[11, 12, 99]"])\n        self.assertIsInstance(results[2], ValueError)\n        self.assertEqual(results[3], "[99]")\n        self.assertEqual(generator.stats[0]["batch_size"], 4)\n        self.assertEqual(generator.stats[0]["output_tokens"], [2, 3, 3, 1])\n        self.tokenizer.pad.assert_called_once_with(\n            [self.tokenizer.return_value] * 4, padding=True, return_tensors="pt")\n        self.model.generate.return_value.__getitem__.assert_called_once_with(\n            (slice(None), slice(2, None)))\n\n    def test_overlong_prompt_fails_only_its_own_row(self):\n        generator = self.load()\n        short = {"input_ids": [7, 8], "attention_mask": [1, 1]}\n        self.tokenizer.side_effect = [short, {"input_ids": [1] * 8193}, short]\n        self.model.generate.return_value.__getitem__.return_value.tolist.return_value = [[99], [99]]\n        results = generator.generate_batch([([], [])] * 3)\n        self.assertEqual(results[0], "Done")\n        self.assertIsInstance(results[1], ValueError)\n        self.assertIn("8193", str(results[1]))\n        self.assertEqual(results[2], "Done")\n        self.tokenizer.pad.assert_called_once_with([short, short], padding=True, return_tensors="pt")\n\n    def test_shared_cuda_failure_is_reported_for_every_affected_row(self):\n        generator = self.load()\n        self.model.generate.side_effect = RuntimeError("CUDA out of memory")\n        results = generator.generate_batch([([], [])] * 4)\n        self.assertEqual(len(results), 4)\n        self.assertTrue(all(isinstance(result, RuntimeError) for result in results))\n        self.model.generate.assert_called_once()\n\n    def test_requires_two_visible_gpus_before_loading(self):\n        self.cuda.device_count = lambda: 1\n        with self.assertRaisesRegex(RuntimeError, "T4 x2"):\n            self.load()\n        self.factory.from_pretrained.assert_not_called()\n\n    def test_rejects_single_gpu_map_and_cpu_offload(self):\n        for mapping in ({"": 0}, {"a": 0, "b": 1, "c": "cpu"}):\n            self.model.hf_device_map = mapping\n            with self.subTest(mapping=mapping), self.assertRaisesRegex(RuntimeError, "must span"):\n                self.load()\n\n    def test_explicit_one_gpu_mode(self):\n        self.model.hf_device_map = {"": 0}\n        generator = self.load(1)\n        self.assertEqual(generator.num_gpus, 1)\n        self.assertEqual(self.factory.from_pretrained.call_args.kwargs["device_map"], {"": 0})\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'requirements.txt': '# Separate environment from the Qwen3 / Transformers 4.x training notebooks.\ntransformers>=5.3,<6\naccelerate>=1.12,<2\ntorch>=2.6\nsafetensors>=0.4.5\nsherpa-onnx>=1.12,<2\nsoundfile>=0.13,<1\nscipy>=1.13,<2\nnumpy>=1.26,<3\n', 'validation_ids.txt': 'train_0001\ntrain_0004\ntrain_0015\ntrain_0016\ntrain_0017\ntrain_0031\ntrain_0063\ntrain_0066\ntrain_0075\ntrain_0080\ntrain_0088\ntrain_0090\ntrain_0095\ntrain_0106\ntrain_0110\ntrain_0117\ntrain_0130\ntrain_0133\ntrain_0141\ntrain_0144\ntrain_0145\ntrain_0147\ntrain_0162\ntrain_0163\ntrain_0164\ntrain_0169\ntrain_0172\ntrain_0181\ntrain_0194\ntrain_0196\ntrain_0203\ntrain_0207\ntrain_0214\ntrain_0216\ntrain_0219\ntrain_0225\ntrain_0228\ntrain_0231\ntrain_0233\ntrain_0236\ntrain_0240\ntrain_0255\ntrain_0278\ntrain_0285\ntrain_0304\ntrain_0313\ntrain_0316\ntrain_0321\ntrain_0330\ntrain_0344\ntrain_0345\ntrain_0346\ntrain_0350\ntrain_0354\ntrain_0357\ntrain_0378\ntrain_0381\ntrain_0395\ntrain_0405\ntrain_0422\ntrain_0435\ntrain_0442\ntrain_0453\ntrain_0457\ntrain_0468\ntrain_0469\ntrain_0476\ntrain_0480\ntrain_0481\ntrain_0486\ntrain_0497\ntrain_0500\ntrain_0505\ntrain_0508\ntrain_0518\ntrain_0536\ntrain_0538\ntrain_0550\ntrain_0554\ntrain_0560\ntrain_0563\ntrain_0572\ntrain_0577\ntrain_0596\ntrain_0598\ntrain_0618\ntrain_0621\ntrain_0624\ntrain_0625\ntrain_0628\ntrain_0664\ntrain_0689\ntrain_0694\ntrain_0719\ntrain_0729\ntrain_0752\ntrain_0764\ntrain_0770\ntrain_0772\ntrain_0777\ntrain_0779\ntrain_0787\ntrain_0790\ntrain_0800\ntrain_0805\ntrain_0814\ntrain_0815\ntrain_0819\ntrain_0820\ntrain_0834\ntrain_0857\ntrain_0864\ntrain_0867\ntrain_0870\ntrain_0875\ntrain_0923\ntrain_0933\ntrain_0935\ntrain_0943\ntrain_0945\ntrain_0947\ntrain_0948\ntrain_0953\ntrain_0958\ntrain_0959\ntrain_0960\ntrain_0977\ntrain_0978\ntrain_0991\ntrain_1000\ntrain_1014\ntrain_1016\ntrain_1019\ntrain_1021\ntrain_1049\ntrain_1060\ntrain_1062\ntrain_1070\ntrain_1081\ntrain_1089\ntrain_1091\ntrain_1096\ntrain_1097\ntrain_1109\ntrain_1117\ntrain_1118\ntrain_1123\ntrain_1124\ntrain_1128\ntrain_1137\ntrain_1147\ntrain_1148\ntrain_1155\ntrain_1160\ntrain_1163\ntrain_1175\ntrain_1180\ntrain_1188\ntrain_1201\ntrain_1206\ntrain_1209\ntrain_1215\ntrain_1216\ntrain_1233\ntrain_1234\ntrain_1246\ntrain_1248\ntrain_1249\ntrain_1253\ntrain_1265\ntrain_1278\ntrain_1289\ntrain_1295\ntrain_1300\ntrain_1305\ntrain_1310\ntrain_1311\ntrain_1317\ntrain_1325\ntrain_1346\ntrain_1351\ntrain_1352\ntrain_1353\ntrain_1355\ntrain_1356\ntrain_1363\ntrain_1373\ntrain_1379\ntrain_1380\ntrain_1384\ntrain_1399\ntrain_1405\ntrain_1408\ntrain_1432\ntrain_1433\ntrain_1437\ntrain_1440\ntrain_1448\ntrain_1458\ntrain_1460\ntrain_1471\ntrain_1472\ntrain_1474\ntrain_1475\ntrain_1476\ntrain_1478\ntrain_1486\ntrain_1491\ntrain_1497\ntrain_1506\ntrain_1512\ntrain_1519\ntrain_1526\ntrain_1528\ntrain_1533\ntrain_1540\ntrain_1542\ntrain_1544\ntrain_1546\ntrain_1556\ntrain_1557\ntrain_1560\ntrain_1571\ntrain_1572\ntrain_1589\ntrain_1598\ntrain_1606\ntrain_1608\ntrain_1615\ntrain_1617\ntrain_1623\ntrain_1653\ntrain_1654\ntrain_1658\ntrain_1661\ntrain_1668\ntrain_1697\ntrain_1702\ntrain_1703\ntrain_1705\ntrain_1707\ntrain_1712\ntrain_1713\ntrain_1715\ntrain_1746\n'}
for name, content in PAYLOAD.items():
    (WORK/name).write_text(content)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(WORK/"requirements.txt")], check=True)
subprocess.run([sys.executable, "-B", str(WORK/"test_agent.py")], check=True)
subprocess.run([sys.executable, "-B", str(WORK/"test_scoring.py")], check=True)
subprocess.run([sys.executable, "-B", str(WORK/"test_placement.py")], check=True)
subprocess.run([sys.executable, "-c", "import torch; print('Visible GPUs:', torch.cuda.device_count()); assert torch.cuda.device_count() >= int(__import__('sys').argv[1]), 'Select GPU T4 x2 in Kaggle settings'", str(NUM_GPUS)], check=True)


In [ ]:
assert PROFILE in {"smoke", "validation", "random200", "public"}
split = "public_test" if PROFILE == "public" else "train"
if split == "public_test":
    assert MODE == "asr", "Set MODE=asr for public test"
assert MODE in {"oracle", "asr"}
input_root = Path("/kaggle/input")
if DATA_ROOT:
    data_root = Path(DATA_ROOT)
else:
    candidates = sorted({p.parent.parent for p in input_root.rglob(f"{split}.jsonl")
                         if p.parent.name == split})
    if not candidates and PROFILE == "random200" and not DATA_ZIP:
        # KaggleHub attaches/downloads the existing private dataset using the
        # notebook's Kaggle identity; no credentials are embedded in source.
        import kagglehub
        dataset = Path(kagglehub.dataset_download("cymeu4l0t/zaloai-dataset"))
        candidates = sorted({p.parent.parent for p in dataset.rglob(f"{split}.jsonl")
                             if p.parent.name == split})
    if not candidates:
        archives = [Path(DATA_ZIP)] if DATA_ZIP else sorted(input_root.rglob("train+public_test.zip"))
        assert len(archives) == 1, "Set DATA_ROOT or DATA_ZIP explicitly"
        extracted = Path("/kaggle/working/lifemate_data")
        assert not extracted.exists(), "Extraction already exists; set DATA_ROOT to it"
        extracted.mkdir()
        with zipfile.ZipFile(archives[0]) as archive:
            for member in archive.infolist():
                target = (extracted/member.filename).resolve()
                assert target.is_relative_to(extracted.resolve()), "Unsafe ZIP path"
            archive.extractall(extracted)
        candidates = sorted({p.parent.parent for p in extracted.rglob(f"{split}.jsonl")
                             if p.parent.name == split})
    assert len(candidates) == 1, f"Ambiguous dataset roots: {candidates}"
    data_root = candidates[0]
assert (data_root/split/"zaic_tools"/"__init__.py").is_file()
output = Path("/kaggle/working")/RUN_NAME
assert not output.exists(), "Change RUN_NAME for a new run"
selection = ["--ids", str(WORK/"validation_ids.txt")] if PROFILE == "validation" else []
if PROFILE == "random200":
    import hashlib, random
    dataset_path = data_root/"train"/"train.jsonl"
    all_ids = sorted(json.loads(line)["id"] for line in dataset_path.read_text().splitlines() if line.strip())
    assert len(all_ids) == len(set(all_ids)), "Duplicate dataset IDs"
    assert 0 < SAMPLE_SIZE <= len(all_ids)
    selected_ids = sorted(random.Random(SEED).sample(all_ids, SAMPLE_SIZE))
    (WORK/"selected_ids.txt").write_text("\n".join(selected_ids) + "\n")
    (WORK/"selection.json").write_text(json.dumps({
        "seed": SEED, "sample_size": SAMPLE_SIZE, "population_size": len(all_ids),
        "dataset_sha256": hashlib.sha256(dataset_path.read_bytes()).hexdigest(),
        "selected_ids": selected_ids}, indent=2))
    selection = ["--ids", str(WORK/"selected_ids.txt")]
    print(f"Selected {SAMPLE_SIZE}/{len(all_ids)} train samples, seed={SEED}")
if PROFILE == "smoke":
    selection += ["--limit", "8"]
common = ["--data-root", str(data_root), "--split", split] + selection
print("DATA_ROOT:", data_root, "PROFILE:", PROFILE, "MODE:", MODE)


In [ ]:
transcript_args = []
if MODE == "asr":
    assert ASR_MODEL_DIR, "Set ASR_MODEL_DIR to the supplied Zipformer ONNX directory"
    transcript_path = Path("/kaggle/working")/(RUN_NAME + "_transcripts.json")
    subprocess.run([sys.executable, "-B", str(WORK/"transcribe.py"), *common,
                    "--model-dir", ASR_MODEL_DIR, "--output", str(transcript_path)], check=True)
    transcript_args = ["--transcripts", str(transcript_path)]
command = [sys.executable, "-B", str(WORK/"run.py"), *common,
           "--mode", MODE, *transcript_args, "--model", MODEL, "--revision", REVISION,
           "--max-input", str(MAX_INPUT), "--max-new-tokens", str(MAX_NEW_TOKENS),
           "--max-steps", str(MAX_STEPS), "--num-gpus", str(NUM_GPUS),
           "--batch-size", str(BATCH_SIZE), "--output", str(output)]
subprocess.run(command, check=True)


In [ ]:
import shutil
if PROFILE == "random200":
    for name in ("selected_ids.txt", "selection.json"):
        shutil.copyfile(WORK/name, output/name)
report = json.loads((output/"report.json").read_text())
if PROFILE == "random200":
    assert report["samples"] == SAMPLE_SIZE
print(json.dumps({k: v for k, v in report.items() if k != "generation"}, indent=2))
# Schema validation of actual submission; labels are never passed to inference.
env = dict(os.environ, PYTHONPATH=str(data_root/split))
subprocess.run([sys.executable, "-m", "zaic_tools", "--split", split,
                "--root", str(data_root), "--validate", str(output/"submission.jsonl")],
               env=env, check=True)
archive = shutil.make_archive(str(output), "zip", output)
print("Artifacts:", archive)
from IPython.display import FileLink, display
display(FileLink(str(Path(archive).relative_to("/kaggle/working"))))
